# Chapter 1 — the tensor engine

What you'll learn:

- how `Tensor` records a computation graph, and how `backward()` replays it
- why gradients accumulate with `+=` (fan-out)
- why broadcasting forces **summation** in backward — the unbroadcast rule
- the matmul VJP as a shape game you can't lose
- how the finite-difference referee keeps everyone honest

Companion: [DERIVATIONS.md](../../DERIVATIONS.md) §0–§5 · [engine/tensor.py](../../engine/tensor.py)
· prereq: the [warmups](../warmups/) (micrograd → bigrams → MLP).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))   # repo root
import numpy as np

from engine.tensor import Tensor, unbroadcast
np.random.seed(0)

The whole engine in four sentences: `Tensor` wraps a numpy array. Every op
returns a new `Tensor` carrying a closure (`_backward`) that knows how to push
gradients to its inputs — a vector–Jacobian product. `backward()` sorts the
graph topologically and runs the closures in reverse. Everything else in the
repo is calling these ops in the right order.

In [2]:
# the smallest possible graph: y = x^2 + x, dy/dx = 2x + 1
a = Tensor(np.array([2.0, 3.0]), requires_grad=True)
b = a * a + a
loss = b.sum()
loss.backward()

print('a.grad          =', a.grad)
print('2a + 1 (by hand) =', 2 * a.data + 1)

a.grad          = [5. 7.]
2a + 1 (by hand) = [5. 7.]


## fan-out: why `+=`

`a` was used twice up there (`a * a`), and its gradient came out right anyway.
A node used in k places receives the **sum** of k contributions — the chain
rule over multiple paths. This is the `b = a + a` check from the micrograd
warmup, now in tensors. It is not a convenience; residual connections and
weight tying are nothing but fan-outs.

In [3]:
a = Tensor(np.array([5.0]), requires_grad=True)
b = a + a
b.backward()
print('a.grad =', a.grad, ' (two paths, 1 + 1)')
assert a.grad[0] == 2.0

a.grad = [2.]  (two paths, 1 + 1)


## broadcasting is copying (§1)

Add a bias `(C,)` to activations `(B, T, C)` and numpy silently reuses each
`bias[c]` at every `(b, t)`. Mathematically that's B·T copies — a B·T-way
fan-out — so the bias gradient must **sum over the broadcast axes**. One rule
to remember: `g_x` always has exactly the shape of `x`.

In [4]:
x = Tensor(np.random.randn(4, 8, 16).astype(np.float32), requires_grad=True)  # (B,T,C)
bias = Tensor(np.zeros(16, dtype=np.float32), requires_grad=True)              # (C,)
(x + bias).sum().backward()

print('bias.grad shape:', bias.grad.shape)          # (16,), not (4,8,16)
assert np.allclose(bias.grad, np.ones((4, 8, 16), dtype=np.float32).sum(axis=(0, 1)))
print('equals g_out summed over (B, T): True')

bias.grad shape: (16,)
equals g_out summed over (B, T): True


`unbroadcast(g, shape)` is the helper every elementwise backward calls. Two
cases (DERIVATIONS §1): axes that were **added** in front get summed away;
axes that were **stretched from size 1** get summed with `keepdims`.

In [5]:
g = np.ones((4, 8, 16))
print('(C,)   <-', unbroadcast(g, (16,)).shape)      # added leading axes summed away
print('(1,8,16) <-', unbroadcast(g, (1, 8, 16)).shape)  # stretched axis summed, kept as 1

(C,)   <- (16,)
(1,8,16) <- (1, 8, 16)


## matmul (§2): the shape game

For `C = A @ B`: `g_A = g_C @ Bᵀ` and `g_B = Aᵀ @ g_C`. You never memorize
this — the required shape forces it. `g_C` is `(m, n)` and `g_A` must be
`(m, k)`: the only way to build `(m, k)` out of `g_C` and `B (k, n)` is
`(m,n) @ (n,k)`. The shape check is the proof sketch.

In [6]:
A = Tensor(np.random.randn(2, 3), requires_grad=True)
B = Tensor(np.random.randn(3, 4), requires_grad=True)
(A @ B).sum().backward()

g = np.ones((2, 4))                       # upstream gradient of sum()
print('g_A == g @ B.T :', np.allclose(A.grad, g @ B.data.T))
print('g_B == A.T @ g :', np.allclose(B.grad, A.data.T @ g))

g_A == g @ B.T : True
g_B == A.T @ g : True


## the referee

Every rule above is also checked *mechanically*: perturb one input entry by
±h, run the forward twice, and `(L(x+h) − L(x−h)) / 2h` must match the
engine's gradient entry. Central differences in float64 with h=1e-5 — float64
because finite differences subtract nearly-equal numbers, which is exactly
where float32 dies. Training runs float32 for speed and memory; checking runs
float64 for truth. The full battery is 25 checks, one per claim in
DERIVATIONS, including the nasty cases: broadcast shapes, repeated embedding
indices, weight-tying fan-out.

In [7]:
import subprocess
r = subprocess.run([sys.executable, 'tests/test_ops.py'],
                   cwd=os.path.abspath('../..'), capture_output=True, text=True)
print(r.stdout[-1500:])
print('exit code:', r.returncode)
assert r.returncode == 0

1: max_rel_err=3.28e-12
[OK  ] const arith input0: max_rel_err=4.03e-11
[OK  ] pow -0.5 (rsqrt) input0: max_rel_err=2.19e-10
[OK  ] exp input0: max_rel_err=7.93e-11
[OK  ] log input0: max_rel_err=1.02e-10
[OK  ] tanh input0: max_rel_err=3.66e-11
[OK  ] matmul 2d input0: max_rel_err=2.45e-11
[OK  ] matmul 2d input1: max_rel_err=6.68e-11
[OK  ] matmul bcast weight input0: max_rel_err=9.48e-11
[OK  ] matmul bcast weight input1: max_rel_err=2.48e-10
[OK  ] matmul batched input0: max_rel_err=5.05e-10
[OK  ] matmul batched input1: max_rel_err=2.34e-10
[OK  ] sum keepdims input0: max_rel_err=1.26e-11
[OK  ] mean keepdims input0: max_rel_err=1.44e-10
[OK  ] sum axis0 chained input0: max_rel_err=2.85e-11
[OK  ] reshape+transpose input0: max_rel_err=4.68e-11
[OK  ] slice input0: max_rel_err=7.78e-11
[OK  ] embedding gather (repeats) input0: max_rel_err=1.23e-10
[OK  ] gelu input0: max_rel_err=3.79e-10
[OK  ] softmax input0: max_rel_err=1.49e-10
[OK  ] softmax input1: max_rel_err=2.34e-11
[OK  ] 

## exercise: label your gradients

A true story from this repo (DERIVATIONS §11): with attention scores
`S = Q @ Kᵀ`, the formula `g_K = Qᵀ @ g_S` was written down at 5am. It looks
right and it is **wrong** — it computes the gradient of `Kᵀ` (the tensor
literally multiplied), not of `K`. The five-second catch: `Qᵀ g_S` has shape
`(hs, T)`, which is Kᵀ's shape, and a gradient must have the shape of its
tensor. Verify both candidates against the engine:

In [8]:
T, hs = 5, 3
q = Tensor(np.random.randn(T, hs), requires_grad=True)
k = Tensor(np.random.randn(T, hs), requires_grad=True)
(q @ k.transpose(0, 1)).sum().backward()   # S = Q @ K^T, upstream g_S = ones

g_s = np.ones((T, T))
print('k.grad shape:', k.grad.shape)
print('g_Sᵀ @ Q  ->', np.allclose(k.grad, g_s.T @ q.data), ' (correct)')
try:
    np.allclose(k.grad, q.data.T @ g_s)
except ValueError as e:
    print('Qᵀ @ g_S  -> ValueError:', e)
    print('           (hs,T) is Kᵀ\'s shape — wrong label, wrong tensor)')

k.grad shape: (5, 3)
g_Sᵀ @ Q  -> True  (correct)
Qᵀ @ g_S  -> ValueError: operands could not be broadcast together with shapes (5,3) (3,5) 
           (hs,T) is Kᵀ's shape — wrong label, wrong tensor)


One memory footnote worth knowing cold: `backward()` **frees each
intermediate's grad right after its own closure runs** — reverse-topological
order guarantees no one needs it anymore, and leaves (parameters) keep theirs
for the optimizer. That single line halved peak GPU memory during the real
run. Same spirit: `exp`/`tanh` cache their forward output and reuse it in
backward, never recompute.

## defend this

1. Why must gradients **sum** at fan-out — what rule of calculus is that?
2. In `x @ W`, the weight is broadcast over the batch. Does the batch-sum in
   `g_W` come from the matmul rule or the broadcast rule? (Trick question —
   check `__matmul__`'s backward in the engine.)
3. Why must embedding backward scatter-**add**? What silently breaks with
   assignment when a token id repeats in a batch?
4. Why float64 + central differences in the checker, but float32 in training?

## your notes

<!-- TODO(you): answers in your own words, plus anything that surprised you.
     this cell is part of your review pass — non-empty before we ship. -->